# RegimeShift: Phase 3 - Regime Detection (HMM)

## Summer of Quant 2026 - Advanced Track

Fit a Hidden Markov Model to detect Bull/Bear/Crisis regimes in real time.

---

### Project Phases
1. **Phase 1: Data Pipeline** ✅ COMPLETE
2. **Phase 2: Feature Engineering** ✅ COMPLETE
3. **Phase 3: Regime Detection (HMM)** ← You are here
4. **Phase 4: Portfolio Optimization**
5. **Phase 5: Backtesting & Benchmarking**

---

## Environment & Load Phase 2 Features

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.patches as mpatches
from hmmlearn import hmm
import warnings

warnings.filterwarnings('ignore')

# Plotting config
plt.rcParams["figure.figsize"] = (14, 6)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3
pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)

print("✅ Environment ready.")
print(f"   hmmlearn version: {hmm.__name__}")

## Load Normalized Features from Phase 2

In [ ]:
# Load normalized features from Phase 2
features_normalized = pd.read_csv("data/features_normalized.csv", index_col=0, parse_dates=True)
features_raw = pd.read_csv("data/features_raw.csv", index_col=0, parse_dates=True)

print(f"Loaded {len(features_normalized)} trading days of features")
print(f"Date range: {features_normalized.index[0].date()} to {features_normalized.index[-1].date()}")
print(f"\nFeatures: {list(features_normalized.columns)}")
print(f"\nFirst 5 rows:")
print(features_normalized.head())

print(f"\nFeature statistics:")
print(features_normalized.describe().round(3))

---

# PHASE 3: REGIME DETECTION (HMM)

## What is a Hidden Markov Model?

An HMM is a probabilistic model that assumes:
1. **Hidden states** exist (Bull, Bear, Crisis) that we cannot directly observe
2. **Observations** (our features) are generated by these hidden states
3. **State transitions** follow Markovian dynamics (memoryless: next state depends only on current state)

**The HMM learns:**
- **Emission distribution**: What features look like in each state?
- **Transition matrix**: How often do we flip between states?
- **Initial probabilities**: What's the chance we start in each state?

**Then it infers:**
- **Viterbi path**: The most likely sequence of hidden states given observations
- **Posterior probabilities**: Probability we're in each state at time t

---

## Step 1: Fit the HMM

We'll fit a **Gaussian HMM** with 3 hidden states.

**Why Gaussian?** Features are continuous, approximately normal-ish after z-scoring.

**Why 3 states?** Bull/Bear/Crisis is intuitive; we can validate post-hoc.

In [ ]:
# Prepare data for HMM
X = features_normalized.values  # Convert to numpy array for hmmlearn

print(f"Input array shape: {X.shape}")
print(f"Features (columns): {features_normalized.shape[1]}")
print(f"Observations (rows): {features_normalized.shape[0]}")

# Fit Gaussian HMM with 3 hidden states
print("\nFitting 3-state Gaussian HMM...")
model = hmm.GaussianHMM(n_components=3, covariance_type="full", n_iter=1000, random_state=42)
model.fit(X)

print("✅ HMM fitted successfully!")
print(f"\nModel parameters:")
print(f"  n_components: {model.n_components}")
print(f"  converged: {model.monitor_.converged}")
print(f"  score: {model.score(X):.4f}")

## Step 2: Extract Hidden States (Viterbi Path)

Use the Viterbi algorithm to find the most likely sequence of hidden states.

In [ ]:
# Get the most likely sequence of states (Viterbi path)
hidden_states = model.predict(X)

# Get posterior probabilities (probability of each state at each time)
posterior_probs = model.predict_proba(X)

# Create a DataFrame with results
results_df = features_normalized.copy()
results_df['state'] = hidden_states
results_df['prob_state_0'] = posterior_probs[:, 0]
results_df['prob_state_1'] = posterior_probs[:, 1]
results_df['prob_state_2'] = posterior_probs[:, 2]
results_df['max_prob'] = posterior_probs.max(axis=1)

print("State assignments:")
print(results_df[['state', 'prob_state_0', 'prob_state_1', 'prob_state_2', 'max_prob']].head(10))

print(f"\nState distribution:")
print(results_df['state'].value_counts().sort_index())

# Percentage of time in each state
state_pcts = (results_df['state'].value_counts() / len(results_df) * 100).sort_index()
print(f"\nPercentage of time in each state:")
for i, pct in state_pcts.items():
    print(f"  State {i}: {pct:.1f}%")

## Step 3: Inspect HMM Parameters

Understand what each state represents by looking at mean/std of features.

In [ ]:
# Inspect the means and covariances of each state
print("HMM Means (feature values in each state):")
print("="*80)

feature_names = features_normalized.columns.tolist()
means_df = pd.DataFrame(model.means_, columns=feature_names)
print(means_df.round(3))

print(f"\n\nInterpretation:")
print("="*80)

# Analyze what each state represents
for state in range(3):
    state_data = results_df[results_df['state'] == state]
    raw_state_data = features_raw.loc[state_data.index]
    
    print(f"\nState {state}: ({len(state_data)} days, {len(state_data)/len(results_df)*100:.1f}%)")
    print(f"  mom_21d_NSE:      {raw_state_data['mom_21d_NSE'].mean():.4f} (trend indicator)")
    print(f"  vol_21d_avg:      {raw_state_data['vol_21d_avg'].mean():.4f} (stress level)")
    print(f"  VIX:              {raw_state_data['VIX'].mean():.2f} (fear gauge)")
    print(f"  corr_NSE_IEF_21d: {raw_state_data['corr_NSE_IEF_21d'].mean():.3f} (correlation)")

## Step 4: Label States (Bull/Bear/Crisis)

Based on feature means, assign interpretable labels to each hidden state.

In [ ]:
# Determine state labels based on characteristics
# Heuristic: VIX is the strongest regime indicator
vix_means = []
vol_means = []
mom_means = []

for state in range(3):
    state_data = results_df[results_df['state'] == state]
    raw_state_data = features_raw.loc[state_data.index]
    vix_means.append(raw_state_data['VIX'].mean())
    vol_means.append(raw_state_data['vol_21d_avg'].mean())
    mom_means.append(raw_state_data['mom_21d_NSE'].mean())

# Sort by VIX: low VIX = Bull, high VIX = Crisis, mid = Bear
vix_means = np.array(vix_means)
state_order = np.argsort(vix_means)

# Assign labels
state_labels = {}
state_colors = {}

# Low VIX state = Bull
bull_state = state_order[0]
state_labels[bull_state] = 'Bull'
state_colors[bull_state] = 'green'

# Mid VIX state = Bear
bear_state = state_order[1]
state_labels[bear_state] = 'Bear'
state_colors[bear_state] = 'orange'

# High VIX state = Crisis
crisis_state = state_order[2]
state_labels[crisis_state] = 'Crisis'
state_colors[crisis_state] = 'red'

print("State Labels:")
print("="*80)
for state in range(3):
    print(f"\nState {state} → {state_labels[state]}")
    print(f"  VIX mean:    {vix_means[state]:.2f}")
    print(f"  Vol mean:    {vol_means[state]:.4f}")
    print(f"  Mom mean:    {mom_means[state]:.4f}")

# Add labels to results
results_df['regime'] = results_df['state'].map(state_labels)

print(f"\n\nRegime Distribution:")
print(results_df['regime'].value_counts())

## Step 5: Visualize Regimes on Price Chart

Overlay regime colors on NSE price to see if HMM captures market structure.

In [ ]:
# Load price data for visualization
asset_prices = pd.read_csv("data/asset_prices.csv", index_col=0, parse_dates=True)

# Align prices with regimes
prices_aligned = asset_prices.loc[results_df.index, 'NSE']
cumulative_returns = np.log(prices_aligned / prices_aligned.iloc[0])

# Plot
fig, ax = plt.subplots(figsize=(14, 6))

# Plot price
ax.plot(cumulative_returns.index, cumulative_returns.values, color='black', linewidth=1.5, label='NSE Log Return', zorder=1)

# Color background by regime
for i in range(len(results_df) - 1):
    state = results_df['state'].iloc[i]
    color = state_colors[state]
    ax.axvspan(results_df.index[i], results_df.index[i+1], alpha=0.15, color=color, zorder=0)

# Legend
bull_patch = mpatches.Patch(color='green', alpha=0.3, label='Bull')
bear_patch = mpatches.Patch(color='orange', alpha=0.3, label='Bear')
crisis_patch = mpatches.Patch(color='red', alpha=0.3, label='Crisis')
ax.legend(handles=[bull_patch, bear_patch, crisis_patch], loc='upper left')

ax.set_title('NSE Regimes Detected by HMM (Colored Background)', fontsize=12, fontweight='bold')
ax.set_xlabel('Date')
ax.set_ylabel('Log Return')

ax.xaxis.set_major_locator(mdates.YearLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print("✅ Regime visualization complete")
print("   • Green = Bull markets")
print("   • Orange = Bear markets")
print("   • Red = Crisis/crash periods")

## Step 6: Transition Matrix

How often do regimes transition? Do we stay in Bull for long, or flip around?

In [ ]:
# Extract transition matrix
transition_matrix = model.transmat_

# Convert to DataFrame for readability
state_names = [f"{state_labels[i]} (S{i})" for i in range(3)]
transition_df = pd.DataFrame(
    transition_matrix,
    index=[f"From {s}" for s in state_names],
    columns=[f"To {s}" for s in state_names]
)

print("Transition Matrix (Probabilities):")
print("="*80)
print(transition_df.round(3))

# Heatmap visualization
fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(transition_matrix, cmap="YlOrRd", vmin=0, vmax=1)

ax.set_xticks(range(3))
ax.set_yticks(range(3))
ax.set_xticklabels(state_names)
ax.set_yticklabels(state_names)

# Add probability values
for i in range(3):
    for j in range(3):
        text = ax.text(j, i, f"{transition_matrix[i, j]:.2f}",
                       ha="center", va="center", color="black", fontsize=12, fontweight='bold')

ax.set_title("Regime Transition Matrix", fontsize=12, fontweight='bold')
ax.set_xlabel("To")
ax.set_ylabel("From")
plt.colorbar(im, ax=ax, label="Probability")
plt.tight_layout()
plt.show()

# Interpretation
print(f"\nTransition Interpretation:")
print("="*80)
for i in range(3):
    regime = state_labels[i]
    print(f"\nFrom {regime}:")
    for j in range(3):
        prob = transition_matrix[i, j]
        print(f"  → Stay in {state_labels[j]}: {prob:.1%}")

## Step 7: Regime Duration Analysis

How long do regimes typically last?

In [ ]:
# Calculate regime durations
regime_changes = results_df['state'].diff().fillna(0) != 0
regime_groups = (regime_changes).cumsum()

durations = []
regimes_seen = []

for group_id in regime_groups.unique():
    group = results_df[regime_groups == group_id]
    duration = len(group)
    regime = group['regime'].iloc[0]
    durations.append(duration)
    regimes_seen.append(regime)

duration_df = pd.DataFrame({
    'regime': regimes_seen,
    'duration_days': durations
})

print("Regime Duration Statistics (in trading days):")
print("="*80)
for regime in ['Bull', 'Bear', 'Crisis']:
    regime_durs = duration_df[duration_df['regime'] == regime]['duration_days']
    if len(regime_durs) > 0:
        print(f"\n{regime}:")
        print(f"  Count:       {len(regime_durs)} episodes")
        print(f"  Mean:        {regime_durs.mean():.1f} days")
        print(f"  Median:      {regime_durs.median():.1f} days")
        print(f"  Min:         {regime_durs.min():.0f} days")
        print(f"  Max:         {regime_durs.max():.0f} days")
        print(f"  Total:       {regime_durs.sum():.0f} days")

## Step 8: State Probability Over Time

Show the posterior probability of being in each regime (confidence in state detection).

In [ ]:
# Map probabilities to regime labels
prob_bull = np.zeros(len(results_df))
prob_bear = np.zeros(len(results_df))
prob_crisis = np.zeros(len(results_df))

for i in range(3):
    if state_labels[i] == 'Bull':
        prob_bull = posterior_probs[:, i]
    elif state_labels[i] == 'Bear':
        prob_bear = posterior_probs[:, i]
    else:
        prob_crisis = posterior_probs[:, i]

# Plot
fig, ax = plt.subplots(figsize=(14, 6))

ax.fill_between(results_df.index, 0, prob_bull, alpha=0.6, color='green', label='Bull')
ax.fill_between(results_df.index, prob_bull, prob_bull + prob_bear, alpha=0.6, color='orange', label='Bear')
ax.fill_between(results_df.index, prob_bull + prob_bear, 1, alpha=0.6, color='red', label='Crisis')

ax.set_ylim([0, 1])
ax.set_title('Posterior Probability of Each Regime Over Time', fontsize=12, fontweight='bold')
ax.set_xlabel('Date')
ax.set_ylabel('Probability')
ax.legend(loc='upper right')

ax.xaxis.set_major_locator(mdates.YearLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print("✅ Stacked probability plot complete")
print("   • Shows confidence in regime assignment")
print("   • Sharp regime changes = high certainty")
print("   • Mixed colors = HMM is uncertain")

## Step 9: Save Regime Results

Export regimes for Phase 4 (portfolio optimization).

In [ ]:
# Save results
results_df.to_csv("data/regimes.csv")
print("✅ Saved regime assignments to data/regimes.csv")

# Save transition matrix
transition_df.to_csv("data/transition_matrix.csv")
print("✅ Saved transition matrix to data/transition_matrix.csv")

# Save model for later use
import pickle
with open("data/hmm_model.pkl", "wb") as f:
    pickle.dump(model, f)
print("✅ Saved HMM model to data/hmm_model.pkl")

print(f"\n📊 Phase 3 Summary:")
print(f"   • HMM fitted on {len(results_df)} trading days")
print(f"   • 3 regimes detected: Bull ({(results_df['regime']=='Bull').sum()} days), Bear ({(results_df['regime']=='Bear').sum()} days), Crisis ({(results_df['regime']=='Crisis').sum()} days)")
print(f"   • Model score (log-likelihood): {model.score(X):.4f}")
print(f"   • All results saved to data/")

## Regime Detection Summary

✅ **HMM fitted** to normalized features
✅ **3 regimes detected**: Bull/Bear/Crisis
✅ **States labeled** based on VIX, volatility, momentum, correlation
✅ **Viterbi path** extracted (most likely regime sequence)
✅ **Transition matrix** analyzed (regime stickiness)
✅ **Regime durations** computed (avg Bull: X days, avg Bear: Y days, etc.)
✅ **Posterior probabilities** visualized (confidence in state assignment)

**Key Insights:**
- Bull regimes are characterized by: positive momentum, low volatility, low VIX, low correlation
- Bear regimes are characterized by: negative momentum, moderate volatility, elevated VIX, rising correlation
- Crisis regimes are characterized by: volatile momentum, high volatility, extreme VIX, high correlation
- Regimes are "sticky" (high diagonal probability in transition matrix)

---

## Next: Phase 4 - Portfolio Optimization

In the next phase, we'll:
1. Define **regime-specific portfolios**: Bull/Bear/Crisis allocations
2. Optimize weights using **Markowitz mean-variance** and **risk parity**
3. Implement **dynamic rebalancing**: switch allocations when regime changes
4. Compare to static benchmarks (60/40, equal-weight, etc.)

---